# Runner: Car Collision Model v1

**Purpose:** Orchestrate full pipeline execution using papermill

**Flow:**
1. Template 01: Data assembly (join master + aux, filter folds)
2. Template 02: Data conditioning (types, nulls, feature engineering)
3. Template 03: Verification (data quality checks)
4. Template 04: Model preparation (train/test split, DMatrix)
5. Template 05: Model training (XGBoost fit, predictions, metrics)

**Memory:** Each template runs in subprocess via papermill, cleans memory after checkpoint

In [1]:
import papermill as pm
import os
import sys
from datetime import datetime
import yaml
from pathlib import Path

# Detect project root and add lib to path
current_dir = Path.cwd()
if current_dir.name == 'runners':
    project_root = current_dir.parent
else:
    project_root = current_dir

# Add lib directory to Python path BEFORE importing
lib_path = str(project_root / 'lib')
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

# Now import
from utils import setup_notebook_environment

## Configuration

In [2]:
# Auto-detect project root (cross-machine compatible)
project_root = setup_notebook_environment()

print(f"Project root: {project_root}")
print(f"Current directory: {os.getcwd()}")

# Experiment configuration (relative to project root)
config_path = "config/car_coll/v1"
config_file = f"{config_path}/config.yaml"

# Load config to get output path
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
experiment_name = cfg['experiment']['name']

print(f"\nExperiment: {experiment_name}")
print(f"Config: {config_path}")
print(f"Output: {output_base}")

Project root: /Users/Mach/dev/aps/code/26Dmodelv1
Current directory: /Users/Mach/dev/aps/code/26Dmodelv1

Experiment: car_coll_v1
Config: config/car_coll/v1
Output: output/car_coll/v1


In [3]:
# Create output directories
os.makedirs(f"{output_base}/notebooks", exist_ok=True)
os.makedirs(f"{output_base}/data", exist_ok=True)
os.makedirs(f"{output_base}/models", exist_ok=True)
os.makedirs(f"{output_base}/results", exist_ok=True)

print("Output directories created")

Output directories created


## Stage Configuration

In [4]:
# Define pipeline stages
stages = [
    {
        'name': 'stage_01',
        'title': 'Data Assembly',
        'template': 'templates/01_data_assembly.ipynb',
        'output': f'{output_base}/notebooks/01_data_assembly.ipynb',
        'enabled': cfg['execution']['stages']['stage_01']
    },
    {
        'name': 'stage_02',
        'title': 'Data Conditioning',
        'template': 'templates/02_data_conditioning.ipynb',
        'output': f'{output_base}/notebooks/02_data_conditioning.ipynb',
        'enabled': cfg['execution']['stages']['stage_02']
    },
    {
        'name': 'stage_03',
        'title': 'Verification',
        'template': 'templates/03_verification.ipynb',
        'output': f'{output_base}/notebooks/03_verification.ipynb',
        'enabled': cfg['execution']['stages']['stage_03']
    },
    {
        'name': 'stage_04',
        'title': 'Model Preparation',
        'template': 'templates/04_model_prep.ipynb',
        'output': f'{output_base}/notebooks/04_model_prep.ipynb',
        'enabled': cfg['execution']['stages']['stage_04']
    },
    {
        'name': 'stage_05',
        'title': 'Model Training',
        'template': 'templates/05_model_training.ipynb',
        'output': f'{output_base}/notebooks/05_model_training.ipynb',
        'enabled': cfg['execution']['stages']['stage_05']
    },
    {
        'name': 'stage_06',
        'title': 'SHAP Dataframe',
        'template': 'templates/06_shap_dataframe.ipynb',
        'output': f'{output_base}/notebooks/06_shap_dataframe.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_06', True)
    },
    {
        'name': 'stage_07',
        'title': 'SHAP Analysis',
        'template': 'templates/07_shap_analysis.ipynb',
        'output': f'{output_base}/notebooks/07_shap_analysis.ipynb',
        'enabled': cfg['execution']['stages'].get('stage_07', True)
    }
]

print("Pipeline stages:")
for stage in stages:
    status = "ENABLED" if stage['enabled'] else "DISABLED"
    print(f"  {stage['name']}: {stage['title']} - {status}")

Pipeline stages:
  stage_01: Data Assembly - ENABLED
  stage_02: Data Conditioning - ENABLED
  stage_03: Verification - ENABLED
  stage_04: Model Preparation - ENABLED
  stage_05: Model Training - ENABLED
  stage_06: SHAP Dataframe - ENABLED
  stage_07: SHAP Analysis - ENABLED


## Execute Pipeline

In [5]:
# Pipeline parameters (injected into each template)
parameters = {
    'config_path': config_path
}

print(f"\nParameters to inject:")
print(f"  config_path: {config_path}")


Parameters to inject:
  config_path: config/car_coll/v1


In [ ]:
# Execute each stage
start_time = datetime.now()
print(f"\n{'='*60}")
print(f"PIPELINE EXECUTION START: {start_time}")
print(f"{'='*60}\n")

results = {}

for stage in stages:
    if not stage['enabled']:
        print(f"\n[SKIP] {stage['name']}: {stage['title']}")
        results[stage['name']] = 'skipped'
        continue
    
    print(f"\n{'#'*60}")
    print(f"# {stage['name'].upper()}: {stage['title'].upper()}")
    print(f"{'#'*60}")
    print(f"Template: {stage['template']}")
    print(f"Output:   {stage['output']}")
    print(f"\nExecuting via papermill...\n")
    
    try:
        stage_start = datetime.now()
        
        pm.execute_notebook(
            input_path=stage['template'],
            output_path=stage['output'],
            parameters=parameters,
            kernel_name='python3'
        )
        
        stage_end = datetime.now()
        duration = (stage_end - stage_start).total_seconds()
        
        print(f"\n✓ {stage['name']} completed in {duration:.1f}s")
        results[stage['name']] = 'success'
        
    except Exception as e:
        stage_end = datetime.now()
        duration = (stage_end - stage_start).total_seconds()
        
        print(f"\n✗ {stage['name']} FAILED after {duration:.1f}s")
        print(f"Error: {str(e)}")
        results[stage['name']] = 'failed'
        
        if cfg['execution']['stop_on_error']:
            print(f"\nStopping pipeline (stop_on_error=true)")
            break

end_time = datetime.now()
total_duration = (end_time - start_time).total_seconds()

print(f"\n{'='*60}")
print(f"PIPELINE EXECUTION END: {end_time}")
print(f"Total duration: {total_duration:.1f}s ({total_duration/60:.1f}m)")
print(f"{'='*60}")


PIPELINE EXECUTION START: 2026-08-06 10:06:37.114424


############################################################
# STAGE_01: DATA ASSEMBLY
############################################################
Template: templates/01_data_assembly.ipynb
Output:   output/car_coll/v1/notebooks/01_data_assembly.ipynb

Executing via papermill...



Executing:   0%|          | 0/17 [00:00<?, ?cell/s]


✓ stage_01 completed in 131.3s

############################################################
# STAGE_02: DATA CONDITIONING
############################################################
Template: templates/02_data_conditioning.ipynb
Output:   output/car_coll/v1/notebooks/02_data_conditioning.ipynb

Executing via papermill...



Executing:   0%|          | 0/13 [00:00<?, ?cell/s]


✓ stage_02 completed in 28.4s

############################################################
# STAGE_03: VERIFICATION
############################################################
Template: templates/03_verification.ipynb
Output:   output/car_coll/v1/notebooks/03_verification.ipynb

Executing via papermill...



Executing:   0%|          | 0/9 [00:00<?, ?cell/s]


✓ stage_03 completed in 4.4s

############################################################
# STAGE_04: MODEL PREPARATION
############################################################
Template: templates/04_model_prep.ipynb
Output:   output/car_coll/v1/notebooks/04_model_prep.ipynb

Executing via papermill...



Executing:   0%|          | 0/9 [00:00<?, ?cell/s]


✓ stage_04 completed in 16.3s

############################################################
# STAGE_05: MODEL TRAINING
############################################################
Template: templates/05_model_training.ipynb
Output:   output/car_coll/v1/notebooks/05_model_training.ipynb

Executing via papermill...



Executing:   0%|          | 0/18 [00:00<?, ?cell/s]

## Summary

In [ ]:
# Print summary
print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)

success_count = sum(1 for v in results.values() if v == 'success')
failed_count = sum(1 for v in results.values() if v == 'failed')
skipped_count = sum(1 for v in results.values() if v == 'skipped')

for stage_name, status in results.items():
    emoji = '✓' if status == 'success' else '✗' if status == 'failed' else '-'
    print(f"{emoji} {stage_name}: {status}")

print(f"\nTotal: {len(results)} stages")
print(f"  Success: {success_count}")
print(f"  Failed:  {failed_count}")
print(f"  Skipped: {skipped_count}")
print("="*60)

if failed_count > 0:
    print("\n⚠️  PIPELINE COMPLETED WITH ERRORS")
elif success_count > 0:
    print("\n✓ PIPELINE COMPLETED SUCCESSFULLY")
else:
    print("\n- NO STAGES EXECUTED")

In [ ]:
# Save execution log
log_data = {
    'experiment': experiment_name,
    'config_path': config_path,
    'start_time': start_time.isoformat(),
    'end_time': end_time.isoformat(),
    'duration_seconds': total_duration,
    'results': results
}

log_file = f"{output_base}/execution_log.yaml"
with open(log_file, 'w') as f:
    yaml.dump(log_data, f, default_flow_style=False)

print(f"\nExecution log saved: {log_file}")